# E4 / RQ3 — Export Top-20 từ checkpoint Drive · SAFE v1

**Không train, không thay đổi E4/runs hoặc E4/recovery_runs/checkpoints.** Dùng lại exporter E1 đã dùng trong E3C Fixed, nhưng staging từng checkpoint E4 và kiểm tra layer/seed/model. Đầu ra nằm ở `experiments/E4/exports/<model>/layers_<L>/seed_<seed>/topk/`.

**Bắt buộc:** chạy thử 1 run (`MAX_NEW_EXPORTS=1`), xem `export_verification.json` và metric reconciliation trước khi chạy toàn bộ. Nếu một checkpoint chưa tồn tại hoặc không khớp provenance, dừng an toàn. Không tự tái sử dụng checkpoint E1.


In [ ]:
# CELL 1 — MOUNT + CONFIG
from pathlib import Path
import os, sys, json, re, csv, math, ast, pickle, hashlib, shutil, subprocess, traceback
from datetime import datetime, timezone
from tempfile import mkdtemp
import numpy as np
import scipy.sparse as sp
import torch
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA_DIR=ROOT/'data/raw/amazon'
E4=ROOT/'experiments/E4'
EXPORT_ROOT=E4/'exports'
MODELS=('lightgcn','lightgcn_plus')
LAYERS=(1,2,3)
SEEDS=(2023,2024,2025)
DATASET='amazon'; K=20
RLMREC_COMMIT='22413752246de3dee8ab0d509f7f7a8889080f95'
MAX_NEW_EXPORTS=1  # smoke test; change to None after first PASS
RESUME=True
STOP_ON_ERROR=True
CHECK_TRAIN_MASK=True
MAX_METRIC_ABS_DIFF=2e-4
assert ROOT.is_dir() and DATA_DIR.is_dir() and E4.is_dir(), 'Drive/dataset/E4 missing'
assert torch.cuda.is_available(), 'Enable Colab GPU for inference'
print('E4 export root:',EXPORT_ROOT,'GPU:',torch.cuda.get_device_name(0))


In [ ]:
# CELL 2 — SOURCE REPOS + PINNED EXPORTER
GROUP_REPO=Path('/content/project_sna_group14')
GROUP_REPO_URL='https://github.com/trungkien1502/project_sna_group14.git'
RLMREC_REPO=Path('/content/RLMRec_E4_export_source')
RLMREC_URL='https://github.com/HKUDS/RLMRec.git'
def checked(cmd,**kwargs):
    print('>', ' '.join(map(str,cmd)))
    return subprocess.run(list(map(str,cmd)),check=True,**kwargs)
if not GROUP_REPO.exists(): checked(['git','clone',GROUP_REPO_URL,GROUP_REPO])
assert subprocess.check_output(['git','-C',str(GROUP_REPO),'remote','get-url','origin'],text=True).strip().rstrip('/')==GROUP_REPO_URL.rstrip('/')
EXPORT_SCRIPT=GROUP_REPO/'src/inference/export_topk.py'
assert EXPORT_SCRIPT.is_file(),EXPORT_SCRIPT
if not RLMREC_REPO.exists(): checked(['git','clone',RLMREC_URL,RLMREC_REPO])
assert subprocess.check_output(['git','-C',str(RLMREC_REPO),'remote','get-url','origin'],text=True).strip().rstrip('/')==RLMREC_URL.rstrip('/')
head=subprocess.check_output(['git','-C',str(RLMREC_REPO),'rev-parse','HEAD'],text=True).strip()
if head!=RLMREC_COMMIT:
    status=subprocess.check_output(['git','-C',str(RLMREC_REPO),'status','--porcelain'],text=True).strip()
    assert not status,'Source dirty; use a clean Colab runtime'
    checked(['git','-C',RLMREC_REPO,'fetch','origin'])
    checked(['git','-C',RLMREC_REPO,'checkout','--detach',RLMREC_COMMIT])
link=RLMREC_REPO/'data/amazon';link.parent.mkdir(parents=True,exist_ok=True)
if link.is_symlink():assert link.resolve()==DATA_DIR.resolve()
elif link.exists():raise RuntimeError('Unexpected non-symlink data/amazon')
else:link.symlink_to(DATA_DIR,target_is_directory=True)
GROUP_COMMIT=subprocess.check_output(['git','-C',str(GROUP_REPO),'rev-parse','HEAD'],text=True).strip()
EXPORTER_SHA256=hashlib.sha256(EXPORT_SCRIPT.read_bytes()).hexdigest()
for flag in ('--root','--model','--seed','--k'):assert flag in EXPORT_SCRIPT.read_text(),flag
print('Exporter sha256:',EXPORTER_SHA256,'group commit:',GROUP_COMMIT)


In [ ]:
# CELL 3 — DATA AND CHECKPOINT MANIFEST (READ ONLY)
with (DATA_DIR/'trn_mat.pkl').open('rb') as f:TRAIN=pickle.load(f)
with (DATA_DIR/'tst_mat.pkl').open('rb') as f:TEST=pickle.load(f)
assert sp.issparse(TRAIN) and sp.issparse(TEST)
TRAIN=TRAIN.tocsr();TEST=TEST.tocsr();TRAIN.sort_indices();TEST.sort_indices()
assert TRAIN.shape==TEST.shape==(11000,9332),TRAIN.shape
TEST_USERS=np.flatnonzero(np.diff(TEST.indptr)>0)
assert len(TEST_USERS)==9860,len(TEST_USERS)
PLAN=[]
for model in MODELS:
 for layer in LAYERS:
  for seed in SEEDS:
   # Recovery checkpoint only. Never silently substitute E1 or old E4 run.
   run=E4/'recovery_runs'/model/f'layers_{layer}'/f'seed_{seed}'
   cp=run/'checkpoints'/f'{model}-amazon-{seed}.pth'
   out=EXPORT_ROOT/model/f'layers_{layer}'/f'seed_{seed}'/'topk'
   PLAN.append(dict(model=model,layer=layer,seed=seed,run_dir=run,checkpoint=cp,topk_dir=out,ready=cp.is_file()))
for t in PLAN:
 print(('READY' if t['ready'] else 'WAIT '),t['model'],'L'+str(t['layer']),t['seed'])
print('Available:',sum(t['ready'] for t in PLAN),'/',len(PLAN))
# Missing checkpoints are skipped, never represented as successful exports.


In [ ]:
# CELL 4 — E3C FIXED QA / STAGING UTILITIES (reused)
def sha256_file(path, chunk=2**20):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json_atomic(path, payload):
    path = Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2),encoding='utf-8')
    os.replace(tmp, path)

def verify_topk(path, verification_path=None):
    path=Path(path)
    assert path.is_file(), f'Missing {path}'
    with np.load(path,allow_pickle=False) as z:
        assert {'user_ids','item_ids','scores'}.issubset(z.files), z.files
        users=z['user_ids']; items=z['item_ids']; scores=z['scores']
    assert users.ndim==1 and users.shape==(len(TEST_USERS),), users.shape
    assert items.shape==(len(TEST_USERS),K), items.shape
    assert scores.shape==items.shape, scores.shape
    assert np.issubdtype(users.dtype,np.integer) and np.issubdtype(items.dtype,np.integer)
    assert np.array_equal(np.sort(users),TEST_USERS), 'Test-user set mismatch'
    assert len(np.unique(users))==len(users), 'Duplicate test users'
    assert items.min()>=0 and items.max()<TRAIN.shape[1], 'Invalid item IDs'
    assert np.isfinite(scores).all(), 'Non-finite ranking scores'
    assert np.all(scores[:,:-1] >= scores[:,1:] - 1e-6), 'Scores are not descending'
    assert np.all(np.sort(items,axis=1)[:,1:] != np.sort(items,axis=1)[:,:-1]), 'Duplicate items in Top-K'
    # User-by-user CSR check; never exclude validation/test positives from candidates.
    if CHECK_TRAIN_MASK:
        for u, ids in zip(users,items):
            row=TRAIN.indices[TRAIN.indptr[u]:TRAIN.indptr[u+1]]
            if np.isin(ids,row).any():
                raise AssertionError(f'Train leakage for user {u}')
    if verification_path is not None:
        v=load_json(verification_path)
        assert v.get('status')=='passed', f'Exporter verification failed: {v}'
    return {'status':'passed','n_test_users':int(len(users)),'K':int(K),
            'n_items':int(TRAIN.shape[1]), 'train_mask_checked':bool(CHECK_TRAIN_MASK),
            'no_duplicates':True,'scores_descending':True,'scores_finite':True}


def prepare_runtime_exporter(stage):
    """Create a private runtime copy; never edit the source repo or E1 on Drive.

    Hardcoded project-root references in the group exporter are rewritten to
    the per-run staging directory. Explicit --root still points to this stage.
    Non-project absolute Drive paths (if any) fail closed, not guessed over.
    """
    stage = Path(stage)
    source = EXPORT_SCRIPT.read_text(encoding='utf-8')
    assert sha256_file(EXPORT_SCRIPT) == EXPORTER_SHA256, 'Group exporter source changed since Cell 2.'

    old_root = str(ROOT)  # /content/drive/MyDrive/Data_SNA_Group14
    new_root = str(stage)
    patched = source.replace(old_root, new_root)
    replaced_count = source.count(old_root)

    # Fail closed if any OTHER absolute Google Drive path survives in executable
    # source. A comment with a Drive example is harmless and can be retained.
    unexpected = [(i, line.strip()) for i, line in enumerate(patched.splitlines(), 1)
                  if '/content/drive/MyDrive/' in line and not line.lstrip().startswith('#')]
    if unexpected:
        first = '\n'.join(f'  line {i}: {line[:180]}' for i, line in unexpected[:12])
        raise RuntimeError(
            'Exporter contains Google Drive path(s) outside the known project ROOT. '
            'Refusing to run to avoid reading/writing another experiment. '\
            + 'Unpatched references:\n' + first
        )

    # Preserve the original __file__ and import search paths by wrapping exec.
    # Running the exporter as a standalone script previously worked for E1.
    runtime_copy = stage / '_e4_runtime_export_topk.py'
    compile(patched, str(EXPORT_SCRIPT), 'exec')
    runtime_copy.write_text(patched, encoding='utf-8')
    # Execute with __file__ and sys.path matching the ORIGINAL group exporter:
    # downstream relative filesystem lookups should behave exactly like E1.
    launcher = stage / '_e4_launch_exporter.py'
    launcher.write_text(
        'import sys\n'
        'from pathlib import Path\n'
        f'original = Path({str(EXPORT_SCRIPT)!r})\n'
        f'patched_path = Path({str(runtime_copy)!r})\n'
        'sys.argv[0] = str(original)\n'
        'sys.path.insert(0, str(original.parent))\n'
        'namespace = {"__name__":"__main__", "__file__":str(original), "__package__":None}\n'
        'exec(compile(patched_path.read_text(encoding="utf-8"), str(original), "exec"), namespace)\n',
        encoding='utf-8'
    )
    note = {'project_root_replaced':old_root,
            'isolated_root':new_root,
            'root_literal_replacements':replaced_count,
            'source_sha256':EXPORTER_SHA256,
            'runtime_source_sha256':sha256_file(runtime_copy),
            'group_commit':GROUP_COMMIT}
    print(f'Exporter isolated: {replaced_count} ROOT literal replacement(s)')
    return launcher, note



def metric_consistency(verification_file, expected):
    v=load_json(verification_file)
    exported=v.get('test_metrics_from_export',{})
    assert exported, 'Exporter did not record test_metrics_from_export; manual accuracy audit required.'
    diffs={}
    for metric in ('recall@10','recall@20','ndcg@10','ndcg@20'):
        if metric in exported:
            diffs[metric]=abs(float(exported[metric])-float(expected[metric]))
    assert {'recall@20','ndcg@20'}.issubset(diffs), f'Missing key metrics in {verification_file}'
    assert max(diffs.values())<=MAX_METRIC_ABS_DIFF, f'Export vs trainer mismatch: {diffs}'
    return diffs

def copy_2phase(src,dst):
    dst=Path(dst);dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_name(dst.name+'.partial')
    shutil.copy2(src,tmp)
    os.replace(tmp,dst)



In [ ]:
# CELL 5 — E4-SPECIFIC PROVENANCE + CHECKPOINT STAGING
import yaml

def parse_run(t):
    run=t['run_dir']; model=t['model'];layer=t['layer'];seed=t['seed']
    status_path=run/'train_status.json'
    verification_path=run/'checkpoint_verification.json'
    cfg_path=run/'config_snapshot.yml'
    logpath=run/'training_console.log'
    for path in (status_path,verification_path,cfg_path,logpath):assert path.is_file(),f'Missing {path}'
    status=load_json(status_path);v=load_json(verification_path)
    assert status.get('status')=='trained_pending_export' and status.get('exit_code')==0,status
    assert status.get('model')==model and int(status.get('layers'))==layer and int(status.get('seed'))==seed,status
    assert status.get('source_commit')==RLMREC_COMMIT,status
    assert v.get('model')==model and int(v.get('layer'))==layer and int(v.get('seed'))==seed,v
    assert v.get('loadable') is True,v
    cp=t['checkpoint'];assert cp.is_file() and cp.stat().st_size>1024,cp
    digest=sha256_file(cp)
    assert status.get('checkpoint_sha256')==digest and v.get('sha256')==digest,'Checkpoint SHA mismatch'
    assert int(status.get('checkpoint_bytes'))==cp.stat().st_size,'Checkpoint size mismatch'
    assert Path(status.get('checkpoint_path')).resolve()==cp.resolve(),'Checkpoint path mismatch'
    cfg=yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
    assert int(cfg['train']['seed'])==seed
    assert int(cfg['model']['amazon']['layer_num'])==layer
    if 'layer_num' in cfg['model']:assert int(cfg['model']['layer_num'])==layer
    if model=='lightgcn_plus':
        assert math.isclose(float(cfg['model']['amazon']['kd_weight']),0.01,abs_tol=1e-12)
    text=logpath.read_text(encoding='utf-8',errors='replace')
    pat=r"Best Epoch\s+(\d+)\.\s+Final test result:\s*\{'recall': array\(\[([^\]]+)\]\),\s*'ndcg': array\(\[([^\]]+)\]\)\}"
    matches=list(re.finditer(pat,text,re.S))
    assert matches,'Cannot parse final test metrics in '+str(logpath)
    m=matches[-1]
    extract=lambda s:[float(x) for x in re.findall(r'[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?',s)]
    recall,ndcg=extract(m.group(2)),extract(m.group(3))
    assert len(recall)==len(ndcg)==3,(recall,ndcg)
    metrics={**{f'recall@{k}':v for k,v in zip((5,10,20),recall)},**{f'ndcg@{k}':v for k,v in zip((5,10,20),ndcg)}}
    # Avoid loading arbitrary checkpoints: trusted group-owned Drive only.
    state=torch.load(cp,map_location='cpu',weights_only=True)
    assert isinstance(state,dict) and state and all(isinstance(x,torch.Tensor) for x in state.values())
    del state
    return dict(status=status,verification=v,config=cfg,metrics=metrics,best_epoch=int(m.group(1)),sha=digest)

def stage_checkpoint(t,parsed):
    stage=Path(mkdtemp(prefix='e4_export_',dir='/content'))
    model=t['model'];seed=t['seed'];layer=t['layer']
    fake=stage/'experiments/E1'/model/f'seed_{seed}'
    (fake/'checkpoints').mkdir(parents=True,exist_ok=True)
    (stage/'data/raw').mkdir(parents=True,exist_ok=True)
    (stage/'data/raw/amazon').symlink_to(DATA_DIR,target_is_directory=True)
    (fake/'checkpoints'/t['checkpoint'].name).symlink_to(t['checkpoint'])
    metrics={'experiment':'E4_Graph_Depth_Recovery','model':model,'dataset':DATASET,
             'seed':seed,'layers':layer,'rlmrec_commit':RLMREC_COMMIT,
             'best_epoch_index':parsed['best_epoch'],'best_epoch_number':parsed['best_epoch']+1,
             'test_metrics':parsed['metrics'],'checkpoint':str(t['checkpoint']),'status':'completed'}
    conf={'experiment':'E4_Graph_Depth_Recovery','model':model,'dataset':DATASET,
          'seed':seed,'layers':layer,'rlmrec_commit':RLMREC_COMMIT,
          'training_config':parsed['config'],'status':'completed'}
    write_json_atomic(fake/'metrics.json',metrics)
    write_json_atomic(fake/'run_config.json',conf)
    return stage

def valid_existing(t,sha):
    out=t['topk_dir'];p=out/'E4_export_provenance.json'
    if not p.is_file():return False
    v=load_json(p)
    if v.get('status')!='passed' or v.get('model')!=t['model'] or v.get('layer')!=t['layer'] or v.get('seed')!=t['seed'] or v.get('checkpoint_sha256')!=sha:return False
    verify_topk(out/'topk_test.npz',out/'export_verification.json')
    return True

def publish(t,stage_out,provenance):
    out=t['topk_dir']
    assert not out.exists(),f'Output already exists; refusing overwrite: {out}'
    out.mkdir(parents=True,exist_ok=False)
    for name in ('topk_test.npz','export_meta.json','export_verification.json'):
        src=stage_out/name
        assert src.is_file(),src
        copy_2phase(src,out/name)
    write_json_atomic(out/'E4_export_provenance.json',provenance)
    return out


In [ ]:
# CELL 6 — EXPORT ONE / ALL AVAILABLE (SAFE RESUME)
STATUS_PATH=EXPORT_ROOT/'E4_export_status.csv'
EXPORT_ROOT.mkdir(parents=True,exist_ok=True)
records=[];new_exports=0
for t in PLAN:
    model,layer,seed=t['model'],t['layer'],t['seed']
    record=dict(model=model,layer=layer,seed=seed,status='pending',output=str(t['topk_dir']),error='')
    stage=None
    try:
        if not t['ready']:
            record['status']='waiting_for_checkpoint'
        else:
            parsed=parse_run(t)
            if RESUME and valid_existing(t,parsed['sha']):
                record['status']='already_verified'
            elif MAX_NEW_EXPORTS is not None and new_exports>=MAX_NEW_EXPORTS:
                record['status']='not_run_limit'
            else:
                assert not t['topk_dir'].exists(),'Existing incomplete output; inspect manually (never overwrite)'
                stage=stage_checkpoint(t,parsed)
                launcher,patch=prepare_runtime_exporter(stage)
                cmd=[sys.executable,str(launcher),'--repo',str(RLMREC_REPO),'--root',str(stage),
                     '--model',model,'--seed',str(seed),'--k',str(K)]
                log=EXPORT_ROOT/model/f'layers_{layer}'/f'seed_{seed}'/'export_console.log'
                log.parent.mkdir(parents=True,exist_ok=True)
                env=os.environ.copy();env['PYTHONPATH']=os.pathsep.join([str(EXPORT_SCRIPT.parent),str(GROUP_REPO),env.get('PYTHONPATH','')])
                print('EXPORT',model,'layer',layer,'seed',seed)
                with log.open('w',encoding='utf-8') as f:
                    proc=subprocess.run(cmd,stdout=f,stderr=subprocess.STDOUT,text=True,env=env)
                if proc.returncode:
                    tail='\n'.join(log.read_text(encoding='utf-8',errors='replace').splitlines()[-60:])
                    raise RuntimeError(f'Exporter failed {proc.returncode}:\n{tail}')
                stage_out=stage/'experiments/predictions'/model/f'seed_{seed}'
                meta=load_json(stage_out/'export_meta.json')
                for field in ('checkpoint','checkpoint_path','checkpoint_file'):
                    recorded=meta.get(field)
                    if isinstance(recorded,str) and recorded.endswith('.pth'):
                        path=Path(recorded)
                        if not path.is_absolute():path=stage/path
                        assert path.resolve()==t['checkpoint'].resolve(),(field,recorded)
                qa=verify_topk(stage_out/'topk_test.npz',stage_out/'export_verification.json')
                diffs=metric_consistency(stage_out/'export_verification.json',parsed['metrics'])
                provenance=dict(experiment='E4_Graph_Depth_RQ3',model=model,layer=layer,seed=seed,
                    checkpoint=str(t['checkpoint']),checkpoint_sha256=parsed['sha'],
                    source_run_dir=str(t['run_dir']),group_exporter_commit=GROUP_COMMIT,
                    group_exporter_sha256=EXPORTER_SHA256,rlmrec_commit=RLMREC_COMMIT,
                    ranking_mask='train_only',validation_masked=False,
                    trainer_metrics=parsed['metrics'],export_vs_trainer_abs_diff=diffs,
                    audit=qa,runtime_exporter_patch=patch,status='passed',
                    created_utc=datetime.now(timezone.utc).isoformat())
                out=publish(t,stage_out,provenance)
                assert sha256_file(t['checkpoint'])==parsed['sha']
                verify_topk(out/'topk_test.npz',out/'export_verification.json')
                record['status']='export_passed';new_exports+=1
                print('PASS',out)
    except Exception as exc:
        record['status']='FAILED';record['error']=str(exc)[:1500]
        print('FAILED',model,layer,seed,record['error'])
        traceback.print_exc(limit=2)
    finally:
        if stage is not None and stage.exists():shutil.rmtree(stage)
        record['updated_utc']=datetime.now(timezone.utc).isoformat()
        records.append(record)
        # Separate audit status only; does not alter source training runs.
        with STATUS_PATH.open('w',newline='',encoding='utf-8') as f:
            writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
    if record['status']=='FAILED' and STOP_ON_ERROR:
        raise RuntimeError('Stopped after failed export; inspect error and export_console.log')
print('New exports:',new_exports,'Status:',STATUS_PATH)


In [ ]:
# CELL 7 — FINAL READ-ONLY SUMMARY
import pandas as pd
summary=pd.read_csv(STATUS_PATH)
print(summary.groupby('status').size().to_string())
print('Passed/verified:',summary.status.isin(['export_passed','already_verified']).sum(),'/18')
print('Waiting:',(summary.status=='waiting_for_checkpoint').sum())
display(summary)
